# Morpho-Algo — SFT Training (QLoRA, fits free T4)

Fine-tunes Phi-4-mini (3.8B) for trading decisions using **4-bit QLoRA** — fits a free Colab **T4 (16GB GPU)** comfortably.

Everything (data + fixed driver) is pulled from Hugging Face `bluemorpholimited/morpho-algo`. LoRA checkpoints are **uploaded back to HF every 250 steps** and the run **auto-resumes** if interrupted.

## 0. Set your Hugging Face write token

Your **bluemorpholimited** HF write token (needs write access to `bluemorpholimited/morpho-algo`).

In [ ]:
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your bluemorpholimited HF write token: ")
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN set, len", len(HF_TOKEN))

In [ ]:
# confirm GPU
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable Runtime>GPU")
if torch.cuda.is_available():
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1))

## 1. Install dependencies

In [ ]:
%pip install -q torch transformers accelerate peft bitsandbytes huggingface_hub numpy

> If this next cell fails with a **CUDA/bitsandbytes** mismatch, run `!pip install -q bitsandbytes --upgrade` first or restart runtime, then re-run.

## 2. Pull the QLoRA driver + data from HF

In [ ]:
import os, urllib.request
REPO = "bluemorpholimited/morpho-algo"
WORK = "/content/morpho_algo"
CKPT = f"{WORK}/checkpoints/lora"
SPLITS = f"{WORK}/p1_splits"
os.makedirs(CKPT, exist_ok=True); os.makedirs(SPLITS, exist_ok=True)
HF = os.environ["HF_TOKEN"]

url = f"https://huggingface.co/{REPO}/resolve/main/src/morpho_sft_lora.py"
req = urllib.request.Request(url, headers={"Authorization": f"Bearer {HF}"})
open(f"{WORK}/morpho_sft_lora.py", "wb").write(urllib.request.urlopen(req).read())
print("driver ->", f"{WORK}/morpho_sft_lora.py")

from huggingface_hub import snapshot_download
for sub in ("datasets/P1_v1/splits/train", "datasets/P1_v1/splits/valid"):
    snapshot_download(repo_id=REPO, repo_type="model", token=HF,
                      allow_patterns=[f"{sub}/**"], local_dir=SPLITS)
print("data ->", SPLITS)

## 3. Reduce dataset for T4 speed (optional)

T4 is ~10x slower than A100. Full 14,460 samples over 2 epochs may take many hours. Run this to train on the **first 3,000** samples (~ manageable). You can increase later once it works.

*(Keep 0 to use the full set.)*

In [ ]:
LIMIT_ROWS = 3000   # set to 0 for the full 14,460

## 4. Run QLoRA SFT

4-bit base + LoRA (r=16). bs=1, grad-accum=16. Every 250 steps the small LoRA adapter is uploaded to HF. Interrupted? Just re-run — resumes from the latest LoRA step.

In [ ]:
import os, subprocess, sys
train = f"{SPLITS}/datasets/P1_v1/splits/train"
valid = f"{SPLITS}/datasets/P1_v1/splits/valid"
assert os.path.isdir(train), f"missing {train}"
cmd = [sys.executable, f"{WORK}/morpho_sft_lora.py",
       "--data-train", train, "--data-valid", valid,
       "--out", CKPT, "--model", "microsoft/Phi-4-mini-instruct",
       "--epochs", "2", "--bs", "1", "--grad-accum", "16",
       "--max-len", "896", "--save-every", "250", "--log-every", "5",
       "--run-tag", "qlorav1", "--repo", REPO, "--resume", "auto",
       "--limit", str(LIMIT_ROWS)]
print("Launching QLoRA SFT...", flush=True)
p = subprocess.run(cmd)
print("SFT exit", p.returncode)

## After training

LoRA adapter is on HF: `bluemorpholimited/morpho-algo/checkpoints/lora/qlorav1/`. Tell the assistant once it finishes so the merge-wide eval + DPO stage can be staged next.